# Step 3 - GAN training and utterance classifier experiments (paper Sec. 3.4.3, 3.5.1-3.5.2, Tables 4-5)

Working notebook for the generation experiments:

* **cGAN** (binary cross-entropy) on 28 x 300 utterance matrices - first cell.
* **Conditional Wasserstein GAN** - second cell (also available as `03b_train_wcgan.py`).
* Generating data with saved generators and building augmented training sets.
* Training the utterance-level classifier (Flatten -> Dense(128) -> Dense(4)) on real / augmented data.

Both GANs are based on Keras-GAN by Erik Linder-Norén (https://github.com/eriklindernoren/Keras-GAN).
Generator and discriminator/critic are fully connected (Dense) networks.
The number of training iterations was changed between runs (500 / 1000 / 2000 for the WGAN, 5000 / 20000 for the cGAN).

Label ids: 0 = Anomic, 1 = Broca, 2 = Conduction, 3 = Control.

*Original file name: `Untitled3.ipynb`. Code cells are unchanged from 2019/2020; outputs were cleared and these markdown notes were added in 2026.*

## Conditional GAN (binary cross-entropy)

In [ ]:
from __future__ import print_function, division

from keras.datasets import mnist
from keras.layers import Input, Dense, Reshape, Flatten, Dropout, multiply
from keras.layers import BatchNormalization, Activation, Embedding, ZeroPadding2D
from keras.layers.advanced_activations import LeakyReLU
from keras.layers.convolutional import UpSampling2D, Conv2D
from keras.models import Sequential, Model
from keras.optimizers import Adam

import matplotlib.pyplot as plt

import numpy as np

import json
import numpy as np


class CGAN():
    def __init__(self):
        # Input shape
        self.img_rows = 28
        self.img_cols = 300
        self.channels = 1
        self.img_shape = (self.img_rows, self.img_cols, self.channels)
        self.num_classes = 260
        self.latent_dim = 100

        optimizer = Adam(0.0002, 0.5)

        # Build and compile the discriminator
        self.discriminator = self.build_discriminator()
        self.discriminator.compile(loss=['binary_crossentropy'],
            optimizer=optimizer,
            metrics=['accuracy'])

        # Build the generator
        self.generator = self.build_generator()

        # The generator takes noise and the target label as input
        # and generates the corresponding digit of that label
        noise = Input(shape=(self.latent_dim,))
        label = Input(shape=(1,))
        img = self.generator([noise, label])

        # For the combined model we will only train the generator
        self.discriminator.trainable = False

        # The discriminator takes generated image as input and determines validity
        # and the label of that image
        valid = self.discriminator([img, label])

        # The combined model  (stacked generator and discriminator)
        # Trains generator to fool discriminator
        self.combined = Model([noise, label], valid)
        self.combined.compile(loss=['binary_crossentropy'],
            optimizer=optimizer)

    def build_generator(self):

        model = Sequential()

        model.add(Dense(256, input_dim=self.latent_dim))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Dense(512))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Dense(1024))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Dense(np.prod(self.img_shape), activation='tanh'))
        model.add(Reshape(self.img_shape))

        model.summary()

        noise = Input(shape=(self.latent_dim,))
        label = Input(shape=(1,), dtype='int32')
        label_embedding = Flatten()(Embedding(self.num_classes, self.latent_dim)(label))

        model_input = multiply([noise, label_embedding])
        img = model(model_input)

        return Model([noise, label], img)

    def build_discriminator(self):

        model = Sequential()

        model.add(Dense(512, input_dim=np.prod(self.img_shape)))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dense(512))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))
        model.add(Dense(512))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))
        model.add(Dense(1, activation='sigmoid'))
        model.summary()

        img = Input(shape=self.img_shape)
        label = Input(shape=(1,), dtype='int32')

        label_embedding = Flatten()(Embedding(self.num_classes, np.prod(self.img_shape))(label))
        flat_img = Flatten()(img)

        model_input = multiply([flat_img, label_embedding])

        validity = model(model_input)

        return Model([img, label], validity)

    def train(self, epochs, batch_size=128, sample_interval=50):

        # Load the dataset
        '''
        (X_train, y_train), (_, _) = mnist.load_data()
        
        data = np.load("aphasia_np_vectors.np.npy")
        data = data.reshape(12020, 154, 300, 1)
        
        with open("data_labels.json") as file:
            f = file.read()
            labels = json.loads(f)
        labels_set = set(labels)
        labels_dict = {}
        c = 0
        for i in labels_set:
            labels_dict[i] = c
            c += 1
        nlabels = []
        for i in labels:
            nlabels.append(labels_dict[i])
        nlabels = np.array(nlabels)
        nlabels = nlabels.reshape(-1,1)
        '''
        # Configure input
        #X_train = (X_train.astype(np.float32) - 127.5) / 127.5
        #X_train = np.expand_dims(X_train, axis=3)
        X_train = np.load("aphasia_np_vectors_patientwise_train.np.npy").reshape(47559, 28, 300, 1)
        #y_train = y_train.reshape(-1, 1)
        y_train = np.load("aphasia_np_labels_num_patientwise_train.np.npy")

        # Adversarial ground truths
        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))

        for epoch in range(epochs):

            # ---------------------
            #  Train Discriminator
            # ---------------------

            # Select a random half batch of images
            idx = np.random.randint(0, X_train.shape[0], batch_size)
            imgs, labels = X_train[idx], y_train[idx]

            # Sample noise as generator input
            noise = np.random.normal(0, 1, (batch_size, 100))

            # Generate a half batch of new images
            gen_imgs = self.generator.predict([noise, labels])

            # Train the discriminator
            d_loss_real = self.discriminator.train_on_batch([imgs, labels], valid)
            d_loss_fake = self.discriminator.train_on_batch([gen_imgs, labels], fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # ---------------------
            #  Train Generator
            # ---------------------

            # Condition on labels
            sampled_labels = np.random.randint(0, 10, batch_size).reshape(-1, 1)

            # Train the generator
            g_loss = self.combined.train_on_batch([noise, sampled_labels], valid)

            # Plot the progress
            print ("%d [D loss: %f, acc.: %.2f%%] [G loss: %f]" % (epoch, d_loss[0], 100*d_loss[1], g_loss))

            # If at save interval => save generated image samples
            if epoch % sample_interval == 0:
                print("New sample")
                #self.sample_images(epoch)

    def sample_images(self, epoch):
        r, c = 2, 5
        noise = np.random.normal(0, 1, (r * c, 100))
        sampled_labels = np.arange(0, 10).reshape(-1, 1)

        gen_imgs = self.generator.predict([noise, sampled_labels])

        # Rescale images 0 - 1
        gen_imgs = 0.5 * gen_imgs + 0.5

        fig, axs = plt.subplots(r, c)
        cnt = 0
        for i in range(r):
            for j in range(c):
                axs[i,j].imshow(gen_imgs[cnt,:,:,0], cmap='gray')
                axs[i,j].set_title("Digit: %d" % sampled_labels[cnt])
                axs[i,j].axis('off')
                cnt += 1
        #fig.savefig("images/%d.png" % epoch)
        #plt.close()


if __name__ == '__main__':
    cgan = CGAN()
    cgan.train(epochs=5000, batch_size=8, sample_interval=200)

## Conditional Wasserstein GAN

In [ ]:
from __future__ import print_function, division

from keras.datasets import mnist
from keras.layers import Input, Dense, Reshape, Flatten, Dropout, multiply
from keras.layers import BatchNormalization, Activation, Embedding, ZeroPadding2D
from keras.layers.advanced_activations import LeakyReLU
from keras.layers.convolutional import UpSampling2D, Conv2D
from keras.models import Sequential, Model
from keras.optimizers import Adam
from keras.optimizers import RMSprop
import keras.backend as K

import matplotlib.pyplot as plt

import numpy as np

import json


class CGAN():
    def __init__(self):
        # Input shape
        self.img_rows = 28
        self.img_cols = 300
        self.channels = 1
        self.img_shape = (self.img_rows, self.img_cols, self.channels)
        self.num_classes = 10
        self.latent_dim = 100

        self.n_critic = 5
        self.clip_value = 0.01
        optimizer = RMSprop(lr=0.00005)

        # Build and compile the discriminator
        self.critic = self.build_critic()
        self.critic.compile(loss=self.wasserstein_loss,
            optimizer=optimizer,
            metrics=['accuracy'])
        
        

        # Build the generator
        self.generator = self.build_generator()

        # The generator takes noise and the target label as input
        # and generates the corresponding digit of that label
        z = Input(shape=(self.latent_dim,))
        label = Input(shape=(1,))
        img = self.generator([z, label])

        # For the combined model we will only train the generator
        self.critic.trainable = False

        # The discriminator takes generated image as input and determines validity
        # and the label of that image
        valid = self.critic([img, label])

        # The combined model  (stacked generator and discriminator)
        # Trains generator to fool discriminator
        self.combined = Model([z, label], valid)
        self.combined.compile(loss=self.wasserstein_loss,
            optimizer=optimizer,
            metrics=['accuracy'])
        
    def wasserstein_loss(self, y_true, y_pred):
        return K.mean(y_true * y_pred)

    def build_generator(self):

        model = Sequential()

        model.add(Dense(256, input_dim=self.latent_dim))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Dense(512))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Dense(1024))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Dense(np.prod(self.img_shape), activation='tanh'))
        model.add(Reshape(self.img_shape))

        model.summary()

        noise = Input(shape=(self.latent_dim,))
        label = Input(shape=(1,), dtype='int32')
        label_embedding = Flatten()(Embedding(self.num_classes, self.latent_dim)(label))

        model_input = multiply([noise, label_embedding])
        img = model(model_input)

        return Model([noise, label], img)

    def build_critic(self):

        model = Sequential()

        model.add(Dense(512, input_dim=np.prod(self.img_shape)))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dense(512))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))
        model.add(Dense(512))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))
        model.add(Dense(1, activation='sigmoid'))
        model.summary()

        img = Input(shape=self.img_shape)
        label = Input(shape=(1,), dtype='int32')

        label_embedding = Flatten()(Embedding(self.num_classes, np.prod(self.img_shape))(label))
        flat_img = Flatten()(img)

        model_input = multiply([flat_img, label_embedding])

        validity = model(model_input)

        return Model([img, label], validity)

    def train(self, epochs, batch_size=128, sample_interval=50):

        # Load the dataset
        
        # Configure input
        #X_train = (X_train.astype(np.float32) - 127.5) / 127.5
        #X_train = np.expand_dims(X_train, axis=3)
        #X_train = np.load("aphasia_np_vectors_patientwise_train.np.npy").reshape(47559, 28, 300, 1)
        X_train = np.load("aphasia_np_vectors_patientwise_train.np.npy")
        X_train = X_train.reshape(len(X_train), 28, 300, 1)
        #y_train = y_train.reshape(-1, 1)
        y_train = np.load("aphasia_np_labels_num_patientwise_train.np.npy")

        # Adversarial ground truths
        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))

        for epoch in range(epochs):

            # ---------------------
            #  Train Discriminator
            # ---------------------

            # Select a random half batch of images
            idx = np.random.randint(0, X_train.shape[0], batch_size)
            imgs, labels = X_train[idx], y_train[idx]

            # Sample noise as generator input
            noise = np.random.normal(0, 1, (batch_size, 100))

            # Generate a half batch of new images
            gen_imgs = self.generator.predict([noise, labels])

            # Train the discriminator
            d_loss_real = self.critic.train_on_batch([imgs, labels], valid)
            d_loss_fake = self.critic.train_on_batch([gen_imgs, labels], fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            for l in self.critic.layers:
                weights = l.get_weights()
                weights = [np.clip(w, -self.clip_value, self.clip_value) for w in weights]
                l.set_weights(weights)
            # ---------------------
            #  Train Generator
            # ---------------------

            # Condition on labels
            sampled_labels = np.random.randint(0, 10, batch_size).reshape(-1, 1)

            # Train the generator
            g_loss = self.combined.train_on_batch([noise, sampled_labels], valid)

            # Plot the progress
            print ("%d [D loss: %f] [G loss: %f]" % (epoch, d_loss[0], g_loss[0]))

            # If at save interval => save generated image samples
            if epoch % sample_interval == 0:
                print("New sample")
                #self.sample_images(epoch)

    def sample_images(self, epoch):
        r, c = 2, 5
        noise = np.random.normal(0, 1, (r * c, 100))
        sampled_labels = np.arange(0, 10).reshape(-1, 1)

        gen_imgs = self.generator.predict([noise, sampled_labels])

        # Rescale images 0 - 1
        gen_imgs = 0.5 * gen_imgs + 0.5

        fig, axs = plt.subplots(r, c)
        cnt = 0
        for i in range(r):
            for j in range(c):
                axs[i,j].imshow(gen_imgs[cnt,:,:,0], cmap='gray')
                axs[i,j].set_title("Digit: %d" % sampled_labels[cnt])
                axs[i,j].axis('off')
                cnt += 1
        #fig.savefig("images/%d.png" % epoch)
        #plt.close()


if __name__ == '__main__':
    cgan = CGAN()
    cgan.train(epochs=1500, batch_size=8, sample_interval=200)

## Scratch cells: checking data shapes, earlier 154-word experiments

In [ ]:
import numpy as np
X_train = np.load("aphasia_np_vectors_patientwise_train.np.npy")

In [ ]:
X_train.shape

In [ ]:
(X_train, y_train), (_, _) = mnist.load_data()

        # Configure input
X_train = (X_train.astype(np.float32) - 127.5) / 127.5
X_train = np.expand_dims(X_train, axis=3)
y_train = y_train.reshape(-1, 1)

In [ ]:
from __future__ import print_function, division

from keras.datasets import mnist
from keras.layers import Input, Dense, Reshape, Flatten, Dropout, multiply
from keras.layers import BatchNormalization, Activation, Embedding, ZeroPadding2D
from keras.layers.advanced_activations import LeakyReLU
from keras.layers.convolutional import UpSampling2D, Conv2D
from keras.models import Sequential, Model
from keras.optimizers import Adam

import matplotlib.pyplot as plt

import numpy as np



In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
import json
import numpy as np

data = np.load("aphasia_np_vectors.np.npy")


In [ ]:
data = data.reshape(12020, 154, 300, 1)

In [ ]:
data.shape, labels.shape

In [ ]:
with open("data_labels.json") as file:
    f = file.read()
    labels = json.loads(f)
labels_set = set(labels)
labels_dict = {}
c = 0
for i in labels_set:
    labels_dict[i] = c
    c += 1
nlabels = []
for i in labels:
    nlabels.append(labels_dict[i])
nlabels = np.array(nlabels)
nlabels = nlabels.reshape(-1,1)

In [ ]:
nlabels.shape

In [ ]:
noise = np.random.normal(0, 1, (28, 100))
gen_imgs = self.generator.predict([noise, labels])

In [ ]:
noise = np.random.normal(0, 1, (28, 100))
noise.shape

## Generating data with a trained generator and building augmented training sets

In [ ]:
y_train = np.load("aphasia_np_labels_full5_train.np.npy")
X_train = np.load("aphasia_np_vectors_full5_train.np.npy").reshape(62393, 28, 300, 1)
idx = np.random.randint(0, X_train.shape[0], 28)
labels = y_train[idx]

In [ ]:
labels.shape

In [ ]:
labels

In [ ]:
labels = np.array([4]*15000)

In [ ]:
noise = np.random.normal(0, 1, (15000, 100))
#gen_imgs = cgan.generator.predict([noise, labels])

In [ ]:
labels = np.array([1]*10000)
noise = np.random.normal(0, 1, (10000, 100))
gen_imgs1 = GANmodel.predict([noise, labels])

In [ ]:
1: 10000, 2: 5000, 3: 5000, 4: 15000

In [ ]:
labels = np.array([2]*5000)
noise = np.random.normal(0, 1, (5000, 100))
gen_imgs2 = GANmodel.predict([noise, labels])

In [ ]:
labels = np.array([3]*5000)
noise = np.random.normal(0, 1, (5000, 100))
gen_imgs3 = GANmodel.predict([noise, labels])

In [ ]:
labels = np.array([4]*15000)
noise = np.random.normal(0, 1, (15000, 100))
gen_imgs4 = GANmodel.predict([noise, labels])

In [ ]:
import numpy as np

labels = np.array([0]*10000)
noise = np.random.normal(0, 1, (10000, 100))
gen_imgs0 = GANmodel.predict([noise, labels])

labels = np.array([1]*10000)
noise = np.random.normal(0, 1, (10000, 100))
gen_imgs1 = GANmodel.predict([noise, labels])

labels = np.array([2]*10000)
noise = np.random.normal(0, 1, (10000, 100))
gen_imgs2 = GANmodel.predict([noise, labels])

labels = np.array([3]*10000)
noise = np.random.normal(0, 1, (10000, 100))
gen_imgs3 = GANmodel.predict([noise, labels])

In [ ]:
new_train_vectors = np.append(train_images, gen_imgs0.reshape(10000, 28, 300), axis=0)
print(1)
new_train_vectors = np.append(new_train_vectors, gen_imgs1.reshape(10000, 28, 300), axis=0)
print(2)
new_train_vectors = np.append(new_train_vectors, gen_imgs2.reshape(10000, 28, 300), axis=0)
print(3)
new_train_vectors = np.append(new_train_vectors, gen_imgs3.reshape(10000, 28, 300), axis=0)
print(4)

new_train_labels = np.append(train_labels,np.array([0]*10000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([1]*10000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([2]*10000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([3]*10000), axis=0)

In [ ]:
import numpy as np
new_train_vectors = np.load("aphasia_np_vectors_full5_GAN10k.np.npy")

In [ ]:
#new_train_labels = np.load("aphasia_np_labels_full5_train.np.npy")
new_train_labels = np.append(new_train_labels,np.array([0]*10000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([1]*10000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([2]*10000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([3]*10000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([4]*10000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([7]*10000), axis=0)

In [ ]:
new_train_vectors = np.append(new_train_vectors, gen_imgs4.reshape(15000, 28, 300), axis=0)
print(1)
new_train_vectors = np.append(new_train_vectors, gen_imgs3.reshape(5000, 28, 300), axis=0)
print(2)
new_train_vectors = np.append(new_train_vectors, gen_imgs2.reshape(5000, 28, 300), axis=0)
print(3)
new_train_vectors = np.append(new_train_vectors, gen_imgs1.reshape(10000, 28, 300), axis=0)
print(4)

In [ ]:
np.save("aphasia_np_vectors_full5_GAN10k.np", new_train_vectors)

In [ ]:
new_train_labels = np.append(train_labels,np.array([4]*15000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([3]*5000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([2]*5000), axis=0)
new_train_labels = np.append(new_train_labels,np.array([1]*10000), axis=0)

In [ ]:
set(train_labels)

## Saving / loading generators

In [ ]:
from keras.models import model_from_json

# Save the weights
cgan.generator.save('cWGAN_generator1_pw_half_WGAN1000_new.h5')
print("Saved model to disk")
 

In [ ]:
import tensorflow as tf
from keras.models import model_from_json
GANmodel = tf.keras.models.load_model("cWGAN_generator1_pw.h5")

## Utterance-level classifier (real vs. real + generated data)

In [ ]:
from __future__ import absolute_import, division, print_function
import json

# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#train_images = np.load("aphasia_np_vectors_full5_train.np.npy")
#train_images = new_train_vectors
test_images = np.load("aphasia_np_vectors_patientwise_test.np.npy")
#train_labels = np.load("aphasia_np_labels_full5_train.np.npy")
#train_labels = new_train_labels
test_labels = np.load("aphasia_np_labels_num_patientwise_test.np.npy")

In [ ]:
test_images = np.load("aphasia_np_vectors_patientwise_train_bal.np.npy")
test_images.shape

In [ ]:
import numpy as np
train_images = np.load("aphasia_np_vectors_patientwise_train.np.npy")
train_labels = np.load("aphasia_np_labels_num_patientwise_train.np.npy")

In [ ]:
train_labels.shape

In [ ]:



seed_value = 1
import os
os.environ['PYTHONHASHSEED'] = str(seed_value)
os.environ['TF_DETERMINISTIC_OPS'] = str(seed_value)
import random
random.seed(seed_value)
import numpy as np
np.random.seed(seed_value)
import tensorflow as tf
tf.set_random_seed(seed_value)
from keras import backend as K
session_conf = tf.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)
sess = tf.Session(graph=tf.get_default_graph(), config=session_conf)
K.get_session()


seed(seed_value)

In [ ]:


model = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 300)),
    keras.layers.Dense(128, activation=tf.nn.relu),
    keras.layers.Dense(11, activation=tf.nn.softmax)
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(train_images, train_labels, shuffle=False, epochs=10, validation_data=(test_images[1:], test_labels))
#model.fit(train_x, train_y,validation_split = 0.1, epochs=50, batch_size=4)
plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()


test_loss, test_acc = model.evaluate(test_images[1:], test_labels)

print('Test accuracy:', test_acc)

In [ ]:

seed_value = 1
import os
os.environ['PYTHONHASHSEED'] = str(seed_value)
os.environ['TF_DETERMINISTIC_OPS'] = str(seed_value)
import random
random.seed(seed_value)
import numpy as np
np.random.seed(seed_value)
import tensorflow as tf
tf.set_random_seed(seed_value)
from keras import backend as K
session_conf = tf.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)
sess = tf.Session(graph=tf.get_default_graph(), config=session_conf)
K.get_session()


seed(seed_value)


model.fit(train_images, train_labels, epochs=1, shuffle=False)


test_loss, test_acc = model.evaluate(test_images[1:], test_labels)

print('Test accuracy:', test_acc)


model.fit(train_images, train_labels, epochs=1, shuffle=False)


test_loss, test_acc = model.evaluate(test_images[1:], test_labels)

print('Test accuracy:', test_acc)

In [ ]:
test_loss, test_acc = model.evaluate(test_images[1:], test_labels)

print('Test accuracy:', test_acc)

## Notes from the runs (kept as written in 2019)

In [ ]:
Without added data - 5 ep: 0.29
With added data 5 ep (balansed): 0.305
With added data (10k each) 5 ep: 0.2925 
With added data (10k each) 6 ep: 0.2916666666666667
With added data (10k each) + balansed: 0.2975
With added data (20k each) + balansed 5 ep: 0.28583333333333333
With added data (20k each) + balansed 6 ep: 0.3

In [ ]:
Patient wise baseline - 5 ep: 0.4450955413405303
Patient wise (10k each) - 5 ep: 0.4551167728339031

## Unused experiment: a TensorFlow CNN adapted from Muhammed Buyukkinaci's tutorial code (not used in the paper)

In [ ]:
"""
Created on Thu Mar 22 22:18:53 2018
@author: Muhammed Buyukkinaci
"""
#import cv2#reading and resizing                 
import numpy as np#arrays         
import os#dealing with directories                  
from random import shuffle #to shuffle data
from tqdm import tqdm#loop progress bar  
from sklearn.metrics import roc_auc_score    
import matplotlib.pyplot as plt # for visualizations
import tensorflow as tf # For tensor operations
import pandas as pd # for manipulating data
import zipfile
import os, sys

#Switching to CPU
if tf.test.gpu_device_name():
    print("GPU isn't gonna be used even if you have")
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
else:
    print("No GPU Found")
    print("CPU is gonna be used")

#HYPERPARAMETERS
# our photos are in the size of (80,80,3)
IMG_SIZE = 80

epochs = 30
step_size = 8
IMG_SIZE_ALEXNET = 227
validating_size = 40
nodes_fc1 = 4096
nodes_fc2 = 4096
output_classes = 4


#X = np.array([i[0] for i in train]).reshape(-1,IMG_SIZE_ALEXNET,IMG_SIZE_ALEXNET,3)
#Y = np.array([i[1] for i in train])

X = np.load("aphasia_np_vectors_patientwise_train.np.npy").reshape(47559, 28, 300, 1)
Y = np.load("aphasia_np_labels_num_patientwise_train.np.npy")


print(X.shape)
print(Y.shape)
#cv_x = np.array([i[0] for i in cv]).reshape(-1,IMG_SIZE_ALEXNET,IMG_SIZE_ALEXNET,3)
#cv_y = np.array([i[1] for i in cv])
#test_x = np.array([i[0] for i in test_data]).reshape(-1,IMG_SIZE_ALEXNET,IMG_SIZE_ALEXNET,3)
#test_y = np.array([i[1] for i in test_data])


test_x = np.load("aphasia_np_vectors_patientwise_test.np.npy").reshape(11776, 28, 300, 1)
test_y = np.load("aphasia_np_labels_num_patientwise_test.np.npy")
print(test_x.shape)
print(test_y.shape)

#print(cv_x.shape)

print(test_x.shape)

steps = len(X)
print(steps)
remaining = steps % step_size

#Resetting graph
tf.reset_default_graph()

#Defining Placeholders
x = tf.placeholder(tf.float32,shape=[None,28,300,1])
y_true = tf.placeholder(tf.float32,shape=[None,output_classes])

##CONVOLUTION LAYER 1
#Weights for layer 1
w_1 = tf.Variable(tf.truncated_normal([11,11,1,96], stddev=0.01))
#Bias for layer 1
b_1 = tf.Variable(tf.constant(0.0, shape=[[11,11,1,96][1]]))
#Applying convolution
c_1 = tf.nn.conv2d(x, w_1,strides=[1, 4, 4, 1], padding='VALID')
#Adding bias
c_1 = c_1 + b_1
#Applying RELU
c_1 = tf.nn.relu(c_1)
								
print(c_1)
##POOLING LAYER1
p_1 = tf.nn.max_pool(c_1, ksize=[1, 3, 3, 1],strides=[1, 2, 2, 1], padding='VALID')
print(p_1)

##CONVOLUTION LAYER 2
#Weights for layer 2
w_2 = tf.Variable(tf.truncated_normal([5,5,96,256], stddev=0.01))
#Bias for layer 2
b_2 = tf.Variable(tf.constant(1.0, shape=[[5,5,96,256][3]]))
#Applying convolution
c_2 = tf.nn.conv2d(p_1, w_2,strides=[1, 1, 1, 1], padding='SAME')
#Adding bias
c_2 = c_2 + b_2
#Applying RELU
c_2 = tf.nn.relu(c_2)

print(c_2)

##POOLING LAYER2
p_2 = tf.nn.max_pool(c_2, ksize=[1, 3, 3, 1],strides=[1, 2, 2, 1], padding='VALID')
print(p_2)

##CONVOLUTION LAYER 3
#Weights for layer 3
w_3 = tf.Variable(tf.truncated_normal([3, 3, 256, 384], stddev=0.01))
#Bias for layer 3
b_3 = tf.Variable(tf.constant(0.0, shape=[[3, 3, 256, 384][3]]))
#Applying convolution
c_3 = tf.nn.conv2d(p_2, w_3,strides=[1, 1, 1, 1], padding='SAME')
#Adding bias
c_3 = c_3 + b_3
#Applying RELU
c_3 = tf.nn.relu(c_3)

print(c_3)

##CONVOLUTION LAYER 4
#Weights for layer 4
w_4 = tf.Variable(tf.truncated_normal([3, 3, 384, 384], stddev=0.01))
#Bias for layer 4
b_4 = tf.Variable(tf.constant(0.0, shape=[[3, 3, 384, 384][3]]))
#Applying convolution
c_4 = tf.nn.conv2d(c_3, w_4,strides=[1, 1, 1, 1], padding='SAME')
#Adding bias
c_4 = c_4 + b_4
#Applying RELU
c_4 = tf.nn.relu(c_4)

print(c_4)

##CONVOLUTION LAYER 5
#Weights for layer 5
w_5 = tf.Variable(tf.truncated_normal([3, 3, 384, 256], stddev=0.01))
#Bias for layer 5
b_5 = tf.Variable(tf.constant(0.0, shape=[[3, 3, 384, 256][3]]))
#Applying convolution
c_5 = tf.nn.conv2d(c_4, w_5,strides=[1, 1, 1, 1], padding='SAME')
#Adding bias
c_5 = c_5 + b_5
#Applying RELU
c_5 = tf.nn.relu(c_5)

print(c_5)

##POOLING LAYER3
p_3 = tf.nn.max_pool(c_5, ksize=[1, 3, 3, 1],strides=[1, 2, 2, 1], padding='VALID')
print(p_3)

#Flattening
flattened = tf.reshape(p_3,[-1,6*6*256])
print(flattened)

##Fully Connected Layer 1
#Getting input nodes in FC layer 1
input_size = int( flattened.get_shape()[1] )
#Weights for FC Layer 1
w1_fc = tf.Variable(tf.truncated_normal([input_size, nodes_fc1], stddev=0.01))
#Bias for FC Layer 1
b1_fc = tf.Variable( tf.constant(1.0, shape=[nodes_fc1] ) )
#Summing Matrix calculations and bias
s_fc1 = tf.matmul(flattened, w1_fc) + b1_fc
#Applying RELU
s_fc1 = tf.nn.relu(s_fc1)

#Dropout Layer 1
hold_prob1 = tf.placeholder(tf.float32)
s_fc1 = tf.nn.dropout(s_fc1,keep_prob=hold_prob1)

print(s_fc1)

##Fully Connected Layer 2
#Weights for FC Layer 2
w2_fc = tf.Variable(tf.truncated_normal([nodes_fc1, nodes_fc2], stddev=0.01))
#Bias for FC Layer 2
b2_fc = tf.Variable( tf.constant(1.0, shape=[nodes_fc2] ) )
#Summing Matrix calculations and bias
s_fc2 = tf.matmul(s_fc1, w2_fc) + b2_fc
#Applying RELU
s_fc2 = tf.nn.relu(s_fc2)
print(s_fc2)

#Dropout Layer 2
hold_prob2 = tf.placeholder(tf.float32)
s_fc2 = tf.nn.dropout(s_fc2,keep_prob=hold_prob1)

##Fully Connected Layer 3
#Weights for FC Layer 3
w3_fc = tf.Variable(tf.truncated_normal([nodes_fc2,output_classes], stddev=0.01))
#Bias for FC Layer 3b3_fc = tf.Variable( tf.constant(1.0, shape=[output_classes] ) )
b3_fc = tf.Variable( tf.constant(1.0, shape=[output_classes] ) )
#Summing Matrix calculations and bias
y_pred = tf.matmul(s_fc2, w3_fc) + b3_fc
#Applying RELU
print(y_pred)

#Defining loss function

cross_entropy = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits_v2(labels=y_true,logits=y_pred))

#Defining objective
train = tf.train.AdamOptimizer(learning_rate=0.00001).minimize(cross_entropy)

#Defining Accuracy
matches = tf.equal(tf.argmax(y_pred,1),tf.argmax(y_true,1))
acc = tf.reduce_mean(tf.cast(matches,tf.float32))

#Initializing weights
init = tf.global_variables_initializer()

#Starting Empty lists to keep results
acc_list = []
auc_list = []
loss_list = []
saver = tf.train.Saver()

config = tf.ConfigProto(device_count = {'GPU': 0})
with tf.Session(config=config) as sess:
    sess.run(init)
    for i in range(epochs):
        for j in range(0,steps-remaining,step_size):
            #Feeding step_size-amount data with 0.5 keeping probabilities on DROPOUT LAYERS
            _,c = sess.run([train,cross_entropy],
			feed_dict={x:X[j:j+step_size] , y_true:Y[j:j+step_size],hold_prob1:0.5,hold_prob2:0.5})
        
        
		#Writing for loop to calculate test statistics. GTX 1050 isn't able to calculate all test data.
        cv_auc_list = []
        cv_acc_list = []
        cv_loss_list = []
        for v in range(0,len(cv_x)-int(len(cv_x) % validating_size),validating_size):
            acc_on_cv,loss_on_cv,preds = sess.run([acc,cross_entropy,tf.nn.softmax(y_pred)],
			feed_dict={x:cv_x[v:v+validating_size] ,y_true:cv_y[v:v+validating_size] ,hold_prob1:1.0,hold_prob2:1.0})
			
            auc_on_cv = roc_auc_score(cv_y[v:v+validating_size],preds)
            cv_acc_list.append(acc_on_cv)
            cv_auc_list.append(auc_on_cv)
            cv_loss_list.append(loss_on_cv)
        acc_cv_ = round(np.mean(cv_acc_list),5)
        auc_cv_ = round(np.mean(cv_auc_list),5)
        loss_cv_ = round(np.mean(cv_loss_list),5)
        acc_list.append(acc_cv_)
        auc_list.append(auc_cv_)
        loss_list.append(loss_cv_)
        print("Epoch:",i,"Accuracy:",acc_cv_,"Loss:",loss_cv_ ,"AUC:",auc_cv_)
    
    test_auc_list = []
    test_acc_list = []
    test_loss_list = []
    for v in range(0,len(test_x)-int(len(test_x) % validating_size),validating_size):
        acc_on_test,loss_on_test,preds = sess.run([acc,cross_entropy,tf.nn.softmax(y_pred)], 
		feed_dict={x:test_x[v:v+validating_size] ,
		y_true:test_y[v:v+validating_size] ,
		hold_prob1:1.0,
		hold_prob2:1.0})
        
        auc_on_test = roc_auc_score(test_y[v:v+validating_size],preds)
        test_acc_list.append(acc_on_test)
        test_auc_list.append(auc_on_test)
        test_loss_list.append(loss_on_test)
    saver.save(sess, os.path.join(os.getcwd(),"CNN_MC.ckpt"))
    test_acc_ = round(np.mean(test_acc_list),5)
    test_auc_ = round(np.mean(test_auc_list),5)
    test_loss_ = round(np.mean(test_loss_list),5)
    print("Test Results are below:")
    print("Accuracy:",test_acc_,"Loss:",test_loss_,"AUC:",test_auc_)


f,ax=plt.subplots(1,3,figsize=(12,3))
pd.Series(acc_list).plot(kind='line',title='Accuracy on CV data',ax=ax[0])
pd.Series(loss_list).plot(kind='line',figsize=(12,7),title='Loss on CV data',ax=ax[1])
pd.Series(auc_list).plot(kind='line',figsize=(12,7),title='AUC on CV data',ax=ax[2])
plt.subplots_adjust(wspace=0.8)
ax[0].set_title('Accuracy on CV data')
ax[1].set_title('Loss on CV data')
ax[2].set_title('AUC on CV data')
plt.show()


'''
#Restoring a pretrained
with tf.Session() as session:
    saver.restore(session, "CNN_MC.ckpt")
    print("Model restored.") 
    print('Initialized')
    k = session.run([tf.nn.softmax(y_pred)], feed_dict={x:test_x[0:64] , hold_prob1:1,hold_prob2:1})

print(np.array(k).shape)

#Reshaping k
k = np.array(k).reshape(64,output_classes)

print(k[0])

pred_labels = []

for i in range(64):
    r = np.round(k[i],3).argmax()
    if r ==0 : pred_labels.append("chair")
    elif r ==1: pred_labels.append("kitchen")
    elif r ==2: pred_labels.append("knife")
    elif r ==3: pred_labels.append("saucepan")

#Multiple images parameters
w=80
h=80
columns = 8
rows = 8


print(images.shape)
'''

In [ ]:
import numpy as np#arrays         
import os#dealing with directories                  
from random import shuffle #to shuffle data
from tqdm import tqdm#loop progress bar  
from sklearn.metrics import roc_auc_score    
import matplotlib.pyplot as plt # for visualizations
import tensorflow as tf # For tensor operations
import pandas as pd # for manipulating data
import zipfile
import os, sys

In [ ]:
#HYPERPARAMETERS

epochs = 5
step_size = 8
validating_size = 40
nodes_fc1 = 4096
nodes_fc2 = 4096
output_classes = 4

In [ ]:
X = np.load("aphasia_np_vectors_patientwise_train.np.npy").reshape(47559, 28, 300, 1)
Y = np.load("aphasia_np_labels_num_patientwise_train.np.npy")



test_x = np.load("aphasia_np_vectors_patientwise_test.np.npy").reshape(11776, 28, 300, 1)
test_y = np.load("aphasia_np_labels_num_patientwise_test.np.npy")

print(X.shape)

print(Y.shape)

print(test_x.shape)

print(test_y.shape)

In [ ]:
import numpy as np
test_x = np.load("aphasia_np_vectors_patientwise_test.np.npy")

In [ ]:
test_x.shape

In [ ]:
test_x.reshape(0, -1).shape

In [ ]:
#Resetting graph
tf.reset_default_graph()

#Defining Placeholders
x = tf.placeholder(tf.float32,shape=[None,28,300,1])
y_true = tf.placeholder(tf.float32,shape=[None,output_classes])

In [ ]:
##CONVOLUTION LAYER 1
#Weights for layer 1
w_1 = tf.Variable(tf.truncated_normal([11,11,1,96], stddev=0.01))
#Bias for layer 1
b_1 = tf.Variable(tf.constant(0.0, shape=[[11,11,1,96][3]]))
#Applying convolution
c_1 = tf.nn.conv2d(x, w_1,strides=[1, 4, 4, 1], padding='VALID')
#Adding bias
c_1 = c_1 + b_1
#Applying RELU
c_1 = tf.nn.relu(c_1)
print(c_1)

In [ ]:
##POOLING LAYER1
p_1 = tf.nn.max_pool(c_1, ksize=[1, 3, 3, 1],strides=[1, 2, 2, 1], padding='VALID')
print(p_1)

In [ ]:
#Flattening
flattened = tf.reshape(p_1,[-1,2*36*96])
print(flattened)

In [ ]:
##Fully Connected Layer 1
#Getting input nodes in FC layer 1
input_size = int( flattened.get_shape()[1] )
#Weights for FC Layer 1
w1_fc = tf.Variable(tf.truncated_normal([input_size, nodes_fc1], stddev=0.01))
#Bias for FC Layer 1
b1_fc = tf.Variable( tf.constant(1.0, shape=[nodes_fc1] ) )
#Summing Matrix calculations and bias
s_fc1 = tf.matmul(flattened, w1_fc) + b1_fc
#Applying RELU
s_fc1 = tf.nn.relu(s_fc1)
print(s_fc1)

In [ ]:
[0]*3

In [ ]:
1000